# 1002 · 01 · Bronze: land the raw data

Bronze is the **faithful copy** of what arrived. No cleaning, no type changes,
nothing thrown away: if something goes wrong later, we can always go back here.

What bronze *does* add is **metadata** about the load itself, so every row says
where it came from and when:

| Column | Meaning |
|---|---|
| `_ingested_at` | When this row was loaded |
| `_source_file` | Which file it came from |
| `_job_run_id` | Which job run loaded it |

The same notebook loads **any** of the three sources: the job tells it which one
with the `source` parameter.

## 1 · Parameters

In [ ]:
%python
# Values sent by the job task. The defaults let you run the notebook by hand.
dbutils.widgets.text("source", "scada")      # turbines, scada or work_orders
dbutils.widgets.text("run_id", "manual")     # the job sends {{job.run_id}}

SOURCE = dbutils.widgets.get("source")
RUN_ID = dbutils.widgets.get("run_id")
CATALOG = "lab1002"

print(f"Loading source '{SOURCE}' (run {RUN_ID})")

## 2 · Read the raw files

Every column is read as **text** (no `inferSchema`): bronze keeps exactly what the
file says. Types are silver's job.

In [ ]:
%python
from pyspark.sql import functions as F

raw = (spark.read
       .option("header", "true")
       .csv(f"/Volumes/{CATALOG}/landing/data/{SOURCE}*.csv"))

print(f"{raw.count()} rows read")
raw.printSchema()

## 3 · Add the load metadata

In [ ]:
%python
bronze = (raw
          .withColumn("_ingested_at", F.current_timestamp())
          .withColumn("_source_file", F.col("_metadata.file_name"))
          .withColumn("_job_run_id", F.lit(RUN_ID)))

## 4 · Write the bronze table

`overwrite`: each run reloads the source in full, so running the job twice never
duplicates bronze. (In lab 1001 you saw the incremental alternative, `COPY INTO`.)

In [ ]:
%python
table = f"{CATALOG}.bronze.{SOURCE}"

(bronze.write
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(table))

print(f"{spark.table(table).count()} rows in {table}")
display(spark.table(table).limit(5))